# Day S7 — 분자진화와 계통수: 거리에서 나무로

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S7.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — p-distance와 Jukes–Cantor 거리

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
BASES = np.array(list("ACGT"))

def evolve(seq, expected_changes_per_site):
    """각 자리가 주어진 확률로 다른 염기로 바뀐 '자손 서열'을 만든다."""
    seq = seq.copy()
    hit = rng.random(len(seq)) < expected_changes_per_site
    for pos in np.where(hit)[0]:
        seq[pos] = rng.choice(BASES[BASES != seq[pos]])
    return seq

# 정답을 아는 가상 계통: ((A,B),(C,D)),E
root = rng.choice(BASES, 300)
ab = evolve(root, 0.06); cd = evolve(root, 0.06)
seqs = {
    "A": evolve(ab, 0.03), "B": evolve(ab, 0.03),
    "C": evolve(cd, 0.05), "D": evolve(cd, 0.05),
    "E": evolve(root, 0.15),
}
names = list(seqs)

def p_distance(x, y):
    return (x != y).mean()                       # 다른 자리의 비율

def jukes_cantor(p):
    return -0.75 * np.log(1 - 4 * p / 3)         # 겹쳐 일어난 치환을 보정

assert jukes_cantor(0.0) == 0
P = pd.DataFrame([[p_distance(seqs[a], seqs[b]) for b in names] for a in names],
                 index=names, columns=names)
JC = jukes_cantor(P) + 0.0                    # + 0.0은 "-0.000" 표시를 없애는 용도
print("p-distance")
print(P.round(3))
print("\nJukes–Cantor distance")
print(JC.round(3))

print("\np가 커질수록 보정도 커진다")
for p in [0.05, 0.20, 0.40, 0.60]:
    print(f"  p = {p:.2f} → d = {jukes_cantor(p):.3f}")

### 예시 2 — UPGMA로 나무 만들기

In [ ]:
import pandas as pd

def upgma(D):
    """거리 행렬(DataFrame)을 받아 Newick 문자열과 합친 순서를 돌려준다."""
    D = D.copy()
    size = {name: 1 for name in D.index}        # 각 묶음에 든 서열 수
    height = {name: 0.0 for name in D.index}    # 각 묶음의 높이(잎은 0)
    steps = []
    while len(D) > 1:
        labels = list(D.index)
        pairs = [(D.loc[x, y], x, y) for i, x in enumerate(labels) for y in labels[i + 1:]]
        d, x, y = min(pairs)                    # 가장 가까운 두 묶음
        h = d / 2                               # 새 마디의 높이 = 거리의 절반
        new = f"({x}:{h - height[x]:.3f},{y}:{h - height[y]:.3f})"
        steps.append((x, y, round(d, 3)))

        others = [z for z in labels if z not in (x, y)]
        new_row = {z: (D.loc[x, z] * size[x] + D.loc[y, z] * size[y]) / (size[x] + size[y])
                   for z in others}             # 묶음 크기로 가중한 평균 거리
        D = D.loc[others, others]
        for z in others:
            D.loc[new, z] = new_row[z]
            D.loc[z, new] = new_row[z]
        D.loc[new, new] = 0.0
        size[new] = size[x] + size[y]
        height[new] = h
    return D.index[0] + ";", steps

D = pd.DataFrame(                       # 예시 1에서 나온 Jukes–Cantor 거리
    [[0.000, 0.055, 0.201, 0.193, 0.280],
     [0.055, 0.000, 0.184, 0.180, 0.256],
     [0.201, 0.184, 0.000, 0.092, 0.275],
     [0.193, 0.180, 0.092, 0.000, 0.275],
     [0.280, 0.256, 0.275, 0.275, 0.000]],
    index=list("ABCDE"), columns=list("ABCDE"))

newick, steps = upgma(D)
for number, (x, y, d) in enumerate(steps, start=1):
    print(f"{number}단계: {x} + {y}  (거리 {d})")
print(newick)

### 예시 3 — bootstrap 지지도

In [ ]:
# 예시 1(seqs, names, p_distance, jukes_cantor)과 예시 2(upgma)를 실행한 노트북에서 이어서 실행
def clades(steps):
    """합쳐진 묶음마다 그 안의 잎 이름 집합을 돌려준다."""
    found = set()
    for x, y, _ in steps:
        leaves = frozenset(ch for ch in x + y if ch in names)
        found.add(leaves)
    return found

def tree_from(columns):
    sub = {name: seqs[name][columns] for name in names}
    P = pd.DataFrame([[p_distance(sub[a], sub[b]) for b in names] for a in names],
                     index=names, columns=names)
    return upgma(jukes_cantor(P) + 0.0)

n_sites = len(seqs["A"])
_, original_steps = tree_from(np.arange(n_sites))
original = clades(original_steps)

N_BOOT = 200
support = {clade: 0 for clade in original}
for _ in range(N_BOOT):
    columns = rng.integers(0, n_sites, n_sites)      # 자리(열)를 복원추출
    _, steps = tree_from(columns)
    for clade in clades(steps):
        if clade in support:
            support[clade] += 1

for clade, count in sorted(support.items(), key=lambda item: (len(item[0]), sorted(item[0]))):
    print(f"{''.join(sorted(clade)):6s} bootstrap 지지도 {count / N_BOOT:.0%}")

## 직접 해보기

3. **분자 시계 계산.** 가상의 설정으로, C와 D가 1,000만 년 전에 갈라졌다고 합시다. Jukes–Cantor 거리 0.092를 써서 치환 속도 `r = K / (2T)`를 구하고, 그 속도로 A–B(0.055)의 분기 시점을 추정합니다.

In [ ]:
r = 0.092 / (2 * 10_000_000)
print(r)
print(0.055 / (2 * r) / 1_000_000, "백만 년")